In [1]:
from pyspark.sql import functions as F

# Bereinigte Daten aus dem Silver-Layer laden
df_silver = spark.table(
    "silver.vehicle_diagnostics_clean"
)

print("Anzahl Silver-Datensätze:", df_silver.count())

display(df_silver)



StatementMeta(, e63f9723-4b4c-464b-85aa-9cf9271c29a1, 3, Finished, Available, Finished, False)

Anzahl Silver-Datensätze: 5


SynapseWidget(Synapse.DataFrame, 4f21b436-6021-4c38-a490-ed367c9aa9c9)

In [2]:
# Gold-KPIs je Steuergerät berechnen

df_gold = (
    df_silver
    .groupBy("ECU")
    .agg(
        F.count("*").alias("TotalTests"),
        F.sum(
            F.when(F.col("TestResult") == "PASS", 1).otherwise(0)
        ).alias("PassedTests"),
        F.sum(
            F.when(F.col("TestResult") == "FAIL", 1).otherwise(0)
        ).alias("FailedTests")
    )
    .withColumn(
        "PassRate",
        F.round(F.col("PassedTests") / F.col("TotalTests") * 100, 2)
    )
    .withColumn(
        "FailRate",
        F.round(F.col("FailedTests") / F.col("TotalTests") * 100, 2)
    )
)

display(df_gold)


StatementMeta(, e63f9723-4b4c-464b-85aa-9cf9271c29a1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 21a1384a-a6b0-4ff8-b061-79916daa2014)

In [3]:
# Gold-Layer – KPI-Ergebnisse dauerhaft speichern

df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.ecu_quality_kpis")

print("Gold-Tabelle erfolgreich gespeichert!")


StatementMeta(, e63f9723-4b4c-464b-85aa-9cf9271c29a1, 5, Finished, Available, Finished, False)

Gold-Tabelle erfolgreich gespeichert!


In [4]:
# Gespeicherte KPIs aus dem Lakehouse laden

df_check = spark.table("gold.ecu_quality_kpis")

display(df_check)

print("Anzahl Steuergeräte:", df_check.count())


StatementMeta(, e63f9723-4b4c-464b-85aa-9cf9271c29a1, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 0f1a23c3-0cd3-4c79-80c3-1673b9b712ea)

Anzahl Steuergeräte: 3
